# Tutorial 02 — Identification of a 5R1C model with the GA pipeline

This notebook unrolls **`rctoolbox.pipeline.runner.run_pipeline`** step by step on a real calibration case so you can see — and tweak — every moving part. Running this notebook end-to-end is equivalent to invoking:

```bash
cd examples/run_identification/five_r1c_eniso13790/AIBF

rct-train \
  --building-config AIBF \
  --data-path data_train.joblib \
  --identification-config identification_config.json \
  --results-directory-path results \
  --seed 23 \
  --stage 1-3
```

**What this notebook covers.**
1. Load building and identification configs.
2. Set the seed and load the prepared train/val batches.
3. Resolve the model plugin from the registry → `RCParameterSpace`, model factory, problem class.
4. Build the GA objective function (`MAE + std_weight·std`).
5. Wire up the `GAOptimizer` + `HeatDemandProblem`.
6. Run the GA and inspect parameter evolution.
7. Score every candidate on the validation split → `dfR`.
8. Apply the selection threshold to extract $\mathcal{R}^*$ (`dfRtopN`).
9. Cluster $\mathcal{R}^*$ with HDBSCAN and summarise each cluster.
10. Score one representative per cluster and inspect the geometry of the final pool.
11. Build the `results.json` payload (save is commented out so this notebook does **not** overwrite the canonical run).


## Imports

Two paths are configurable: `CONFIG_PATH` points at the identification JSON (the same one `rct-train` reads via `--identification-config`) and `DATA_PATH` points at the prepared joblib (the same one passed to `--data-path`). The building config is loaded by name (`AIBF`, `VUBC`, `P8EE`) from `rctoolbox.configs.buildings/`.


In [ ]:
import json
import os
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from rctoolbox.utils.project_root import find_project_root

project_root = find_project_root()
os.chdir(project_root)
# print(f"Current working directory: {os.getcwd()}")

## 1. Define building ID and configuration; Upload and prepare data

In [ ]:
HEMS_ID = "AIBF"

from rctoolbox.utils.config_loader import load_default_building_config
building_config = load_default_building_config(HEMS_ID)
building_config

In [ ]:
DATA_DIR = project_root / "examples" / "data"
DATA = pd.read_pickle(DATA_DIR / f"{HEMS_ID}_data.pkl")

DATA["w2425"]["df"].columns
dv = "w2425"
df = DATA[dv]["df"]
df.index.min(), df.index.max()

### Train-Val Splits

In [ ]:
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

### TRAIN-VAL TIME-WINDOW SPLITS
wdh = 24; spb = int(1e10); ssh=1;
pre_window_lag = timedelta(hours=24)

t1_train = datetime(2024,12,1)
t2_train = datetime(2024,12,22)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

t1_val = datetime(2024,12,22)
t2_val = datetime(2025,1,1)
tws_val = generate_time_windows(start_time=t1_val, end_time=t2_val, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train, "val": tws_val}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

# PREPARING BATCHES

# ---- TRAIN DATA --- #
# Forecasting batches
t_train        = dataloader.batches["train"]["batch_0"]["t_array"]
T_i_train      = dataloader.batches["train"]["batch_0"]["data_inputs"]["y1"]
T_a_train      = dataloader.batches["train"]["batch_0"]["data_inputs"]["T_a"]
Phi_sol_train  = dataloader.batches["train"]["batch_0"]["data_inputs"]["Phi_sol"]
Phi_HC_train   = dataloader.batches["train"]["batch_0"]["data_inputs"]["Phi_HC"]
Phi_HC_0_train = Phi_HC_train[:,0]
T_i_set_train  = dataloader.batches["train"]["batch_0"]["data_inputs"]["y1_set"]

T_i_0 = T_i_train[:,0]; T_s_0 = T_i_0 - 1; T_m_0 = T_i_0 - 2
y_0_train = np.stack([T_m_0, T_s_0, T_i_0], axis=1) # default y_0

# State initialization batches
t_train_       = dataloader.batches["train"]["batch_0"]["data_inputs"]["t_array_1Dlag"]
T_i_train_     = dataloader.batches["train"]["batch_0"]["data_inputs"]["y1_1Dlag"]
T_a_train_     = dataloader.batches["train"]["batch_0"]["data_inputs"]["T_a_1Dlag"]
Phi_sol_train_ = dataloader.batches["train"]["batch_0"]["data_inputs"]["Phi_sol_1Dlag"]
Phi_HC_train_  = dataloader.batches["train"]["batch_0"]["data_inputs"]["Phi_HC_1Dlag"]
T_i_set_train_ = dataloader.batches["train"]["batch_0"]["data_inputs"]["y1_set_1Dlag"]

T_i_0_ = T_i_train_[:,0]; T_s_0_ = T_i_0_ - 1; T_m_0_ = T_i_0_ - 2
y_0_train_ = np.stack([T_m_0_, T_s_0_, T_i_0_], axis=1)

data_for_init_state_train = {"y_0": y_0_train, # default approximated initial state
                            # pre-window lag inputs for getting initial state from the model
                            "y_0_": y_0_train_, "t_": t_train_, "T_i_": T_i_train_, "T_a_": T_a_train_, "Phi_sol_": Phi_sol_train_, 
                            "Phi_HC_": Phi_HC_train_, "T_i_set_": T_i_set_train_}

train_data = {"t": t_train, "T_i": T_i_train, "T_a": T_a_train, "Phi_sol": Phi_sol_train, "Phi_HC": Phi_HC_train, "Phi_HC_0": Phi_HC_0_train, "T_i_set": T_i_set_train,
              "data_for_init_state": data_for_init_state_train}

# ---- VAL DATA --- #
# Forecasting batches
t_val        = dataloader.batches["val"]["batch_0"]["t_array"]
T_i_val      = dataloader.batches["val"]["batch_0"]["data_inputs"]["y1"]
# T_i_val      = dataloader.batches["val"]["batch_0"]["data_inputs"][y1_set_col]
T_a_val      = dataloader.batches["val"]["batch_0"]["data_inputs"]["T_a"]
Phi_sol_val  = dataloader.batches["val"]["batch_0"]["data_inputs"]["Phi_sol"]
Phi_HC_val   = dataloader.batches["val"]["batch_0"]["data_inputs"]["Phi_HC"]
Phi_HC_0_val = Phi_HC_val[:,0]
T_i_set_val  = dataloader.batches["val"]["batch_0"]["data_inputs"]["y1_set"]

T_i_0 = T_i_val[:,0]; T_s_0 = T_i_0 - 1; T_m_0 = T_i_0 - 2
y_0_val = np.stack([T_m_0, T_s_0, T_i_0], axis=1) # default y_0

# State initialization batches
t_val_       = dataloader.batches["val"]["batch_0"]["data_inputs"]["t_array_1Dlag"]
T_i_val_     = dataloader.batches["val"]["batch_0"]["data_inputs"]["y1_1Dlag"]
T_a_val_     = dataloader.batches["val"]["batch_0"]["data_inputs"]["T_a_1Dlag"]
Phi_sol_val_ = dataloader.batches["val"]["batch_0"]["data_inputs"]["Phi_sol_1Dlag"]
Phi_HC_val_  = dataloader.batches["val"]["batch_0"]["data_inputs"]["Phi_HC_1Dlag"]
T_i_set_val_  = dataloader.batches["val"]["batch_0"]["data_inputs"]["y1_set_1Dlag"]

T_i_0_ = T_i_val_[:,0]; T_s_0_ = T_i_0_ - 1; T_m_0_ = T_i_0_ - 2
y_0_val_ = np.stack([T_m_0_, T_s_0_, T_i_0_], axis=1)

data_for_init_state_val = {"y_0": y_0_val, # default approximated initial state
                            # pre-window lag inputs for getting initial state from the model
                            "y_0_": y_0_val_, "t_": t_val_, "T_i_": T_i_val_, "T_a_": T_a_val_, "Phi_sol_": Phi_sol_val_, ""
                            "Phi_HC_": Phi_HC_val_, "T_i_set_": T_i_set_val_}

val_data = {"t": t_val, "T_i": T_i_val, "T_a": T_a_val, "Phi_sol": Phi_sol_val, "Phi_HC": Phi_HC_val, "Phi_HC_0": Phi_HC_0_val, "T_i_set": T_i_set_val,
            "data_for_init_state": data_for_init_state_val}


data = {"train": train_data, "val": val_data}

print('train arrays:')
for k, v in train_data.items():
    print(f'  {k:>20}: shape={getattr(v, "shape", "dict")}')
print('val arrays:')
for k, v in val_data.items():
    print(f'  {k:>20}: shape={getattr(v, "shape", "dict")}')

## 2. Set configs

`runner.py` loads two JSON blobs: a **building config** (geometry, max HP power) and an **identification config** (model init, GA hyperparameters, selection threshold, clustering hyperparameters). The identification config also carries a `model_type` field that the plugin registry uses to pick the model class (see step 4).


In [ ]:
metric = "MAE"
model_config        = {"model_type": "5R1C_heat_demand", "init_config": {"method": "regime_aware", "variant": "last"}}
optimization_config = {"objective_function": metric, "std_weight": 0.0, "pop_size": 100, "n_gen": 150}
selection_config    = {"criteria": "val error", "threshold_config": {"type": "idx", "x": 12000}}
clustering_config   = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "val error"}

identification_config = {"model_config": model_config, 
                         "optimization_config": optimization_config, 
                         "selection_config": selection_config, 
                         "clustering_config": clustering_config}

In [ ]:
SEED = 23
from rctoolbox.utils.set_global_seed import set_global_seed
set_global_seed(SEED)

## 3. Resolve the model plugin

`rctoolbox` is registry-driven: the `model_type` string in the identification config (`"5R1C_heat_demand"` here) maps to a **plugin** that bundles three things:

| Attribute              | What it is                                                       |
|------------------------|------------------------------------------------------------------|
| `parameter_space_cls`  | Class that turns a building config into RC parameter bounds.     |
| `model_factory`        | Callable that builds a configured model instance.                |
| `problem_cls`          | Pymoo `Problem` subclass with `N_CONSTR` physics constraints.    |

For 5R1C the parameter space is `RCParameterSpace` (10 constraints derived from floor area, n_floors, etc.), the factory is `build_5r1c_model`, and the problem is `HeatDemandProblem`.


In [ ]:
from rctoolbox.pipeline.registry import get_plugin

plugin = get_plugin(model_config['model_type'])

parameter_space = plugin.parameter_space_cls.from_building_config(building_config)
model           = plugin.model_factory(building_config, model_config)

print('plugin:', plugin)
print('free parameters:', list(parameter_space.free_param_bounds.keys()))
print('parameter bounds:')
for p, b in parameter_space.free_param_bounds.items():
    print(f'  {p:>12}: [{b[0]:.2f}, {b[1]:.2f}]')
print('n_constr:', plugin.problem_cls.N_CONSTR)


## 4. Build the objective function

The GA minimises a single scalar per candidate, computed across all training batches:

$$F(\theta) = \mathrm{mean}_b \big[\,\mathrm{err}_b(\theta)\,\big] + w_\sigma \cdot \mathrm{std}_b\big[\,\mathrm{err}_b(\theta)\,\big]$$

- `objective_function` names the per-batch error (`MAE`, `RMSE`, `CV(RMSE)`, ...); a `file.py:function` reference would select a custom objective instead.
- `std_weight` ($w_\sigma$) penalises candidates that fit *on average* but vary wildly across windows. With $w_\sigma = 0$ (this case) the GA chases lowest mean error only.


In [ ]:
from rctoolbox.identification.objective import resolve_objective

objective_fn = resolve_objective(
    optimization_config['objective_function'],
    std_weight=optimization_config['std_weight'],
)

print('objective :', optimization_config['objective_function'])
print('std_weight:', optimization_config['std_weight'])

## 5. Wire up the GA optimizer + problem

The `GAOptimizer` glues the parameter space, the data, and the objective. The `HeatDemandProblem` is the pymoo problem: each candidate is a 7-vector $\theta$, the objective is $F(\theta)$, and there are 10 physics constraints (e.g. floor capacity, conductance ratios). The GA uses LHS sampling for the initial population, SBX crossover, and PM mutation.


In [ ]:
from rctoolbox.identification.ga_optimizer import GAOptimizer

optimizer = GAOptimizer(parameter_space, train_data, val_data, objective_function=objective_fn)

problem = plugin.problem_cls(
    model=model,
    parameter_space=parameter_space,
    data=train_data,
    objective_function=objective_fn,
    n_constr=plugin.problem_cls.N_CONSTR,
)

pop_size = optimization_config['pop_size']
n_gen    = optimization_config['n_gen']

optimizer.set_optimisation_parameters(pop_size=pop_size, n_gen=n_gen)
optimizer.set_model_and_problem(model, problem)

print(f'GA → pop_size={pop_size}, n_gen={n_gen} → {pop_size*n_gen} evaluations')

## I) OPTIMIZATION
---


## 6. Run the GA

This is the running step. The GA history (every candidate of every generation, with its objective and constraint violations) is retained on `optimizer.result` so we can inspect convergence next.

> Tip: for a quick smoke test, set `pop_size=30, n_gen=20` in the config and re-run the previous cell before this one.


In [ ]:
optimizer.run_optimization(seed=SEED)
print('done.')

### Parameter evolution across generations

For each of the 7 free parameters this shows the population median (red line) and the 5–95 % interval (shaded). Convergence shows up as the band narrowing toward a stable value.


In [ ]:
from rctoolbox.evaluation.plotting import plot_free_param_evolution
plot_free_param_evolution(optimizer, show=True, fig_path=None)


## II) SELECTION
---

## 8. Score every candidate on validation → `dfR`

`evaluate_on_validation` walks the entire GA history, replays each candidate against the validation batches and stores the result in `optimizer.dfR` — one row per candidate with `params`, `train_error`, `validation_error`, `generation` and the constraint columns. If no validation data is provided, `evaluate_on_train_only` populates `dfR` from training errors only.


In [ ]:
if val_data is not None:
    optimizer.evaluate_on_validation(use_train_err_threshold=False)
else:
    optimizer.evaluate_on_train_only()

print('dfR shape:', optimizer.dfR.shape)
optimizer.dfR.head()

## 9. Select the top candidates → $\mathcal{R}^*$

Selection is a two-step procedure:

1. **Pick a `criteria` column** — `"val error"`, `"train error"`, or `"train + val error"`.
2. **Apply `threshold_config`** — one of `{"type": "percentage", "x": ratio}`, `{"type": "elbow"}`, `{"type": "idx", "x": n}`.

`select_best_solutions` returns the filtered, criteria-sorted DataFrame (`dfRtopN` = $\mathcal{R}^*$), the numeric threshold value and the cutoff index for plotting.

> **Note on `select_R_star`.** In the pipeline, picking `best_candidate` (step 10) is always done in stage 2, but the R\* cutting shown here is **optional** — controlled by `selection_config.select_R_star` (default `true`). Setting it to `false` skips the two-step procedure above: only `best_candidate = argmin(criteria)` over the full `dfR` is computed, no `dfRtopN` is produced, and stage 3 (clustering) is rejected before any work runs.


In [ ]:
from rctoolbox.identification.selection import select_best_solutions
from rctoolbox.identification.ga_optimizer import add_criteria_column

criteria = selection_config['criteria']
dfR_ = add_criteria_column(optimizer.dfR, criteria)

threshold_config = selection_config['threshold_config']
dfRtopN, threshold, threshold_idx = select_best_solutions(dfR_, threshold_config=threshold_config)

optimizer.dfRtopN = dfRtopN
print(f'selected {len(dfRtopN)} / {len(dfR_)} candidates  (threshold = {threshold:.3f})')
dfRtopN.head()

### Selection diagnostic figure

Three panels: (left) train-vs-val scatter of every GA candidate coloured by generation; (middle) histogram of the chosen criteria with the threshold; (right) sorted criteria curve with the accepted region (sand) and the rejected region (red).


In [ ]:
from rctoolbox.evaluation.plotting import plot_criteria_distribution_and_selection
plot_criteria_distribution_and_selection(
    dfR_,
    dfRtopN,
    criteria=criteria,
    threshold=threshold,
    threshold_idx=threshold_idx,
    threshold_config=threshold_config,
    show=True,
    fig_path=None,
)


## 10. Pick the best candidate + distribution stats over $\mathcal{R}^*$

Stage 2 of the pipeline runs two logically independent operations back-to-back:

- **`compute_best_candidate(criteria)`** sets `optimizer.best_candidate` — the argmin of the chosen `criteria` across the entire GA history `dfR`. This is what gets written under `results.best_candidate` in `results.json`. It is independent of $\mathcal{R}^*$ (the argmin over `dfRtopN` equals the argmin over the full `dfR`), so it **always** runs — even when `select_R_star=false` and step 9 above is skipped.
- **`get_final_model()`** computes distribution statistics (`final_models`, `final_model_dist`, `final_model_average`) over $\mathcal{R}^*$ only. In the pipeline this runs **only** when `select_R_star=true` (i.e. when step 9 produced a `dfRtopN`).


In [ ]:
optimizer.compute_best_candidate(criteria)
optimizer.get_final_model()
optimizer.best_candidate

## III) CLUSTERING
---

## 11. Cluster $\mathcal{R}^*$

The selected pool typically forms several local optima of the loss surface. HDBSCAN groups them so each cluster is a distinct parameter regime. `min_cluster_size` is auto-set as `min_samples_ratio · |R*|` rounded to the order of magnitude.

`apply_clustering_config(0)` writes a `best_cluster` column on `dfRtopN`. If HDBSCAN fails to find any clusters, the whole pool is treated as a single cluster.


In [ ]:
from rctoolbox.utils.get_scale import get_scale

x = len(optimizer.dfRtopN) * clustering_config['min_samples_ratio']
x = int(round(x / get_scale(x)) * get_scale(x))
print(f'min_cluster_size = {x}')

try:
    optimizer.run_clustering_analysis(min_cluster_sizes=[x])
    optimizer.apply_clustering_config(0)
except Exception:
    optimizer.dfRtopN['best_cluster'] = -1

if 'best_cluster' not in optimizer.dfRtopN.columns or (optimizer.dfRtopN['best_cluster'] == -1).all():
    print('No clusters found — treating full dfRtopN as single cluster (label=0).')
    optimizer.dfRtopN['best_cluster'] = 0

optimizer.process_identified_clusters()
print(f'clusters found: {len(optimizer.cluster_summaries)}')
optimizer.dfRtopN['best_cluster'].value_counts().sort_index()


### Distribution and cluster ranges

For each free parameter: (row 0) KDE of $\mathcal{R}^*$, (row 1) cluster ranges vs the building-derived initial bounds, (row 2) cluster ranges only. Useful to see whether clusters separate cleanly along certain parameters.


In [ ]:
from rctoolbox.evaluation.plotting import plot_distribution_and_cluster_ranges
plot_distribution_and_cluster_ranges(optimizer, show=True, fig_path=None)

## 12. Score one representative per cluster

`evaluate_clusters` picks the best candidate within each cluster (by the configured metric/criteria) and stores it in `cluster_summaries[k]['best_candidate']`. The plot compares per-cluster train and validation errors plus the cluster sizes.


In [ ]:
optimizer.evaluate_clusters(
    metric=clustering_config['cl_eval_metric'],
    criteria=clustering_config['cl_eval_criteria'],
)

from rctoolbox.evaluation.plotting import plot_cluster_performance
plot_cluster_performance(optimizer, show=True, fig_path=None)


### Geometry of the final pool

Heatmap shows pairwise correlations between parameters across $\mathcal{R}^*$ (a strong correlation means those two parameters trade off). The PCA projection shows whether clusters separate in the leading PCs.


In [ ]:
from rctoolbox.evaluation.plotting import plot_free_param_heatmap, plot_PCA_of_final_models
plot_free_param_heatmap(optimizer, show=True, fig_path=None)
plot_PCA_of_final_models(optimizer, show=True, fig_path=None)


## 13. Build the `results.json` payload

`runner.py` writes `results.json` (plus `dfRtopN.csv` and, optionally, `optimizer.pkl`) into `--results-directory-path`. The dict below has the same shape. The save line is **commented out** so this tutorial does not overwrite the canonical run already in `cal_for_baseline/v1/AIBF/dec23/results/`.


In [ ]:
def _json_safe(obj):
    if isinstance(obj, float) and obj != obj:  # NaN
        return None
    return obj

results = {
    'setup': {
        'building_config': building_config,
        'seed': SEED,
        'identification_config': identification_config,
    },
    'results': {
        'best_candidate': optimizer.best_candidate,
        'clustering-results': getattr(optimizer, 'cluster_summaries', None),
    },
}

print(json.dumps(results['results']['best_candidate'], indent=2, default=_json_safe))
print('clusters:', list((results['results']['clustering-results'] or {}).keys()))

# To persist the run alongside the rct-train CLI output, uncomment:
# OUT_DIR = CASE_DIR / 'results_notebook'
# OUT_DIR.mkdir(exist_ok=True)
# with open(OUT_DIR / 'results.json', 'w') as f:
#     json.dump(results, f, indent=4, default=_json_safe)
# optimizer.dfRtopN.to_csv(OUT_DIR / 'dfRtopN.csv', index=False)
# with open(OUT_DIR / 'optimizer.pkl', 'wb') as f:
#     pickle.dump(optimizer, f)


## Where to go next

- **Use the calibrated model for forecasting** → `examples/01_predict_with_5R1C.ipynb`.
- **Re-run from the CLI** → 
```bash
cd examples/run_identification/five_r1c_eniso13790/AIBF

rct-train \
  --building-config AIBF \
  --data-path data_train.joblib \
  --identification-config identification_config.json \
  --results-directory-path results \
  --seed 23 \
  --stage 1-3
```
- **Iterate on selection only** → after running once with `--save-optimizer`, re-run with `--stage 2` to try different `threshold_config` values without paying for the GA again.
